<a href="https://colab.research.google.com/github/miriamamin1213-ux/Ablation_studies/blob/main/HECKTORGPT2LoRA3Fold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# HECKTOR GPT-2 LoRA
# HPV Status Prediction | Compact Single-Cell Notebook
# ============================================================

# 1. IMPORT LIBRARIES, SET SEEDS AND SELECT DEVICE
import os, random, gdown
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, balanced_accuracy_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)
import torch
import transformers
import peft

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

!pip uninstall -y torchao


SEED = 42
def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# 2. LOAD HECKTOR EXCEL DATA
DATA_FILE = "HPV2025.xlsx"
if not os.path.exists(DATA_FILE):
    downloaded = gdown.download(
        id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",
        output=DATA_FILE, quiet=False
    )
    if downloaded is None or not os.path.exists(DATA_FILE):
        raise FileNotFoundError("Could not download HPV2025.xlsx.")

df = pd.read_excel(DATA_FILE)
TARGET = "HPV Status"
FEATURES = [
    "Age", "Gender", "Tobacco Consumption", "Alcohol Consumption",
    "Performance Status", "T-stage", "N-stage", "M-stage"
]
missing = [c for c in [TARGET] + FEATURES if c not in df.columns]
if missing:
    raise KeyError(f"Required Excel columns missing: {missing}")

# 3. ROBUSTLY NORMALISE TARGET AND SELECT FEATURES
def normalize_target(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"0", "0.0", "negative", "neg", "false", "no"}:
        return 0
    if s in {"1", "1.0", "positive", "pos", "true", "yes"}:
        return 1
    try:
        n = float(s)
        if n in (0, 1):
            return int(n)
    except (ValueError, TypeError):
        pass
    return np.nan

df["HPV_Label"] = df[TARGET].apply(normalize_target)
invalid_count = int(df["HPV_Label"].isna().sum())
df = df.dropna(subset=["HPV_Label"]).copy()
df["HPV_Label"] = df["HPV_Label"].astype(int)
df = df[FEATURES + ["HPV_Label"]].copy()

if df.empty:
    raise ValueError("No recognised HPV labels found; check the target column.")
counts = df["HPV_Label"].value_counts().reindex([0, 1], fill_value=0)
if (counts < 2).any():
    raise ValueError(f"Insufficient examples for both HPV classes: {counts.to_dict()}")

print("Dataset:", df.shape, "| Invalid/missing target rows dropped:", invalid_count)
print("HPV classes (0=negative, 1=positive):\n", counts)

# 4. STRATIFIED 3-FOLD CROSS-VALIDATION

from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=SEED
)

folds = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(df[FEATURES], df["HPV_Label"]), start=1
):
    train_df = df.iloc[train_idx].copy()
    val_df = df.iloc[val_idx].copy()

    folds.append((train_df, val_df))

    print(f"\nFold {fold}")
    print("Training samples:", len(train_df))
    print("Validation samples:", len(val_df))
    print("Training class counts:",
          train_df["HPV_Label"].value_counts().sort_index().to_dict())
    print("Validation class counts:",
          val_df["HPV_Label"].value_counts().sort_index().to_dict())

print("\n3-fold cross-validation splits created successfully.")


# 5. IMPUTE MISSING FEATURES USING TRAINING DATA ONLY
def impute_fold(train_df, val_df):
    train_df = train_df.copy()
    val_df = val_df.copy()
    fill_values = {}
    for col in FEATURES:
        if pd.api.types.is_numeric_dtype(train_df[col]):
            value = train_df[col].median()
            if pd.isna(value):
                value = 0
        else:
            modes = train_df[col].dropna().astype(str).mode()
            value = modes.iloc[0] if not modes.empty else "Unknown"
        fill_values[col] = value
        for split in (train_df, val_df):
            split[col] = split[col].fillna(value)
            if split[col].dtype == object:
                split[col] = split[col].astype(str).replace(
                    {"": "Unknown", "nan": "Unknown", "None": "Unknown"}
                )
    return train_df, val_df


# 6. CONVERT CLINICAL FEATURES INTO TEXT PROMPTS
def display_value(x):
    if pd.isna(x):
        return "Unknown"
    if isinstance(x, (float, np.floating)) and float(x).is_integer():
        return str(int(x))
    return str(x).strip()

def format_stage(x, prefix):
    value = display_value(x)
    if value == "Unknown" or value.upper().startswith(prefix):
        return value
    try:
        n = float(value)
        if n.is_integer():
            return f"{prefix}{int(n)}"
    except ValueError:
        pass
    return f"{prefix}{value}"

def patient_to_text(row):
    labels = {
        "Age": "Age", "Gender": "Gender",
        "Tobacco Consumption": "Tobacco consumption",
        "Alcohol Consumption": "Alcohol consumption",
        "Performance Status": "Performance status",
        "T-stage": "Tumour T stage", "N-stage": "Nodal N stage",
        "M-stage": "Metastatic M stage"
    }
    parts = ["Q: Predict this patient's HPV status from the following HECKTOR clinical information."]
    for col in FEATURES:
        value = display_value(row[col])
        if col in ("T-stage", "N-stage", "M-stage"):
            value = format_stage(row[col], col[0])
        parts.append(f"{labels[col]}: {value}.")
    parts.append("Is this patient HPV positive? Answer Yes or No.\nA:")
    return " ".join(parts)


# 7. LOAD TOKENIZER AND SET TRAINING PARAMETERS
MODEL_NAME = "openai-community/gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
BATCH_SIZE, EPOCHS = 64, 500
LEARNING_RATE, WEIGHT_DECAY, DROPOUT = 1e-3, 1e-4, 0.1


# 8. BUILD DATASETS AND DATA LOADERS
class HPVTextDataset(Dataset):
    def __init__(self, texts, labels):
        self.items = []
        for text, label in zip(texts, labels):
            enc = tokenizer(
                text, max_length=MAX_LENGTH, padding="max_length",
                truncation=True, return_tensors="pt"
            )
            self.items.append((
                enc["input_ids"].squeeze(0),
                enc["attention_mask"].squeeze(0), int(label)
            ))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        ids, mask, label = self.items[i]
        return {"input_ids": ids, "attention_mask": mask,
                "label": torch.tensor(label, dtype=torch.long)}

loader_args = {
    "batch_size": BATCH_SIZE, "num_workers": 0,
    "pin_memory": torch.cuda.is_available()
}


# ============================================================
# 9. INITIALISE GPT-2 CLASSIFIER AND CLASS-WEIGHTED LOSS
#     UPDATED: LoRA integration only
# ============================================================

from peft import LoraConfig, get_peft_model, TaskType

class HPVNetGPT2Text(nn.Module):
    def __init__(self):
        super().__init__()

        # Load the original GPT-2 backbone
        self.gpt2 = GPT2Model.from_pretrained(MODEL_NAME)
        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = tokenizer.pad_token_id

        # Apply LoRA to GPT-2 attention projections
        lora_config = LoraConfig(
            r=6,
            lora_alpha=12,
            target_modules=["c_attn"],
            lora_dropout=0.0,
            bias="none",
            task_type=TaskType.FEATURE_EXTRACTION
        )

        self.gpt2 = get_peft_model(self.gpt2, lora_config)

        # Keep the original HECKTOR classification head unchanged
        hidden_size = self.gpt2.config.hidden_size
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )

        # Keep the original last-token pooling
        hidden_states = outputs.last_hidden_state
        last_positions = attention_mask.sum(dim=1) - 1

        pooled_output = hidden_states[
            torch.arange(
                hidden_states.size(0),
                device=hidden_states.device
            ),
            last_positions
        ]

        return self.classifier(pooled_output)


# Keep the original weighted loss
def weighted_loss(logits, labels, class_weights):
    losses = nn.functional.cross_entropy(
        logits, labels, reduction="none"
    )
    return (losses * class_weights[labels]).mean()


# ============================================================
# 10. VALIDATION SCORING AND THRESHOLD SELECTION
# ============================================================

def get_scores(model, loader):
    model.eval()
    scores, labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids, mask = batch["input_ids"].to(device), batch["attention_mask"].to(device)
            logits = model(ids, mask)
            scores.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
            labels.extend(batch["label"].numpy())
    return np.asarray(scores), np.asarray(labels, dtype=int)

def evaluate_loss(model, loader, class_weights):
    model.eval()
    total, n_total = 0.0, 0
    with torch.no_grad():
        for b in loader:
            ids, mask = b["input_ids"].to(device), b["attention_mask"].to(device)
            labels = b["label"].to(device)
            loss = weighted_loss(model(ids, mask), labels, class_weights)
            n = len(labels); total += loss.item() * n; n_total += n
    return total / max(n_total, 1)

def choose_threshold(scores, labels):
    thresholds = np.unique(np.r_[0.0, scores, 1.0])
    values = [(float(t), balanced_accuracy_score(labels, scores >= t)) for t in thresholds]
    return max(values, key=lambda x: (x[1], -abs(x[0] - 0.5)))


# ============================================================
# 11. TRAIN MODEL AND SAVE BEST VALIDATION CHECKPOINTS
# ============================================================

OUT_DIR = Path("ROC_HECKTOR_GPT2_CLASSIFIER")
OUT_DIR.mkdir(parents=True, exist_ok=True)

results, predictions_by_model = [], {}
all_history = []

for fold_number, (fold_train_df, fold_val_df) in enumerate(folds, start=1):

    print(f"\n========== FOLD {fold_number}/3 ==========")

    train_df, val_df = impute_fold(fold_train_df, fold_val_df)

    y_train = train_df["HPV_Label"].to_numpy(dtype=int)
    y_val = val_df["HPV_Label"].to_numpy(dtype=int)

    for name, y in [("Train", y_train), ("Validation", y_val)]:
        print(f"{name}: n={len(y)}, class counts={np.bincount(y, minlength=2)}")

    train_texts = [patient_to_text(r) for _, r in train_df.iterrows()]
    val_texts = [patient_to_text(r) for _, r in val_df.iterrows()]

    lengths = [len(tokenizer.encode(t, add_special_tokens=False)) for t in train_texts]
    MAX_LENGTH = min(1024, max(32, int(np.ceil(max(lengths) / 8) * 8)))

    print("Max prompt length:", max(lengths), "| MAX_LENGTH:", MAX_LENGTH)

    train_ds = HPVTextDataset(train_texts, y_train)
    val_ds = HPVTextDataset(val_texts, y_val)

    generator = torch.Generator().manual_seed(SEED + fold_number)

    train_loader = DataLoader(train_ds, shuffle=True, generator=generator, **loader_args)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_args)

    # Initialise a fresh model for each fold
    model = HPVNetGPT2Text().to(device)
    model.gpt2.print_trainable_parameters()

    # Calculate class weights using training data only
    class_counts = np.bincount(y_train, minlength=2)
    class_weights = torch.tensor(
        len(y_train) / (2 * np.maximum(class_counts, 1)),
        dtype=torch.float32,
        device=device
    )

    print(
        "Class weights [negative, positive]:",
        class_weights.detach().cpu().numpy()
    )

    print(
        "Total trainable parameters (LoRA + classifier):",
        sum(p.numel() for p in model.parameters() if p.requires_grad)
    )

    # Optimise only LoRA adapters and the classification head
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    fold_dir = OUT_DIR / f"Fold_{fold_number}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    paths = {
        "Validation_Loss": fold_dir / "best_model_validation_loss.pth",
        "Validation_ROC_AUC": fold_dir / "best_model_validation_roc_auc.pth",
        "Validation_PR_AUC": fold_dir / "best_model_validation_pr_auc.pth",
        "Validation_Balanced_Accuracy": fold_dir / "best_model_validation_balanced_accuracy.pth"
    }

    best = {
        "Validation_Loss": float("inf"),
        "Validation_ROC_AUC": -float("inf"),
        "Validation_PR_AUC": -float("inf"),
        "Validation_Balanced_Accuracy": -float("inf")
    }

    best_info, history = {}, []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum, n_sum = 0.0, 0

        for b in train_loader:
            ids, mask = b["input_ids"].to(device), b["attention_mask"].to(device)
            labels = b["label"].to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = weighted_loss(model(ids, mask), labels, class_weights)
            loss.backward()
            optimizer.step()
            n = len(labels); loss_sum += loss.item() * n; n_sum += n

        train_loss = loss_sum / max(n_sum, 1)
        val_loss = evaluate_loss(model, val_loader, class_weights)
        val_scores, val_labels = get_scores(model, val_loader)
        val_auc = roc_auc_score(val_labels, val_scores)
        val_ap = average_precision_score(val_labels, val_scores)
        threshold, val_ba = choose_threshold(val_scores, val_labels)

        metrics = {
            "Validation_Loss": val_loss,
            "Validation_ROC_AUC": val_auc,
            "Validation_PR_AUC": val_ap,
            "Validation_Balanced_Accuracy": val_ba
        }

        history.append({
            "Fold": fold_number,
            "Epoch": epoch,
            "Train_Loss": train_loss,
            **metrics,
            "Validation_Threshold": threshold
        })

        for name, value in metrics.items():
            improved = value < best[name] if name == "Validation_Loss" else value > best[name]
            if improved:
                best[name] = float(value)
                best_info[name] = {
                    "epoch": epoch,
                    "value": float(value),
                    "threshold": threshold
                }
                torch.save(model.state_dict(), paths[name])

        if epoch % 50 == 0:
            print(f"Fold {fold_number} | Epoch {epoch}/{EPOCHS} | Train Loss={train_loss:.4f} | "
                  f"Val Loss={val_loss:.4f} | ROC-AUC={val_auc:.4f} | "
                  f"PR-AUC={val_ap:.4f} | BA={val_ba:.4f} | Threshold={threshold:.4f}")

    all_history.extend(history)

    pd.DataFrame(history).to_csv(
        fold_dir / "training_history.csv", index=False
    )

    pd.DataFrame([
        {"Checkpoint": k, **v, "Path": str(paths[k])} for k, v in best_info.items()
    ]).to_csv(
        fold_dir / "best_validation_checkpoints.csv", index=False
    )

    # ========================================================
    # 12. EVALUATE ALL CHECKPOINTS FOR EACH FOLD
    # ========================================================

    for name, path in paths.items():
        model.load_state_dict(torch.load(path, map_location=device))
        scores, labels = get_scores(model, val_loader)
        threshold = best_info[name]["threshold"]
        preds = (scores >= threshold).astype(int)
        auc, ap = roc_auc_score(labels, scores), average_precision_score(labels, scores)
        ba = balanced_accuracy_score(labels, preds)
        f1 = f1_score(labels, preds, zero_division=0)
        cm = confusion_matrix(labels, preds, labels=[0, 1])

        print(f"\n=== Fold {fold_number} | {name} | Best epoch {best_info[name]['epoch']} | Threshold {threshold:.4f} ===")
        print(classification_report(
            labels, preds, labels=[0, 1],
            target_names=["HPV Negative", "HPV Positive"], digits=4, zero_division=0
        ))
        print("Confusion matrix [[TN, FP], [FN, TP]]:\n", cm)
        print(f"Validation ROC-AUC={auc:.4f} | PR-AUC={ap:.4f} | BA={ba:.4f} | F1={f1:.4f}")

        results.append({
            "Dataset": "HECKTOR",
            "Model": "GPT-2 Classifier with LoRA",
            "Fold": fold_number,
            "Checkpoint_Selected_By": name,
            "Best_Epoch": best_info[name]["epoch"],
            "Validation_Threshold": threshold,
            "Test_ROC_AUC": auc,
            "Test_PR_AUC": ap,
            "Test_Balanced_Accuracy": ba,
            "Test_F1": f1,
            "TN": cm[0, 0],
            "FP": cm[0, 1],
            "FN": cm[1, 0],
            "TP": cm[1, 1]
        })

        fold_name = f"Fold_{fold_number}_{name}"
        predictions_by_model[fold_name] = (labels, scores, preds, threshold)

        pd.DataFrame({
            "True_Label": labels,
            "HPV_Positive_Score": scores,
            "Predicted_Label": preds,
            "Threshold": threshold
        }).to_csv(
            fold_dir / f"validation_predictions_{name.lower()}.csv", index=False
        )

# Save individual fold results
fold_results_df = pd.DataFrame(results)
fold_results_df.to_csv(OUT_DIR / "three_fold_individual_results.csv", index=False)

# Save combined training history
pd.DataFrame(all_history).to_csv(
    OUT_DIR / "three_fold_training_history.csv", index=False
)

# Calculate mean performance across the three folds for each checkpoint
results_df = fold_results_df.groupby(
    ["Dataset", "Model", "Checkpoint_Selected_By"], as_index=False
).agg({
    "Best_Epoch": "mean",
    "Validation_Threshold": "mean",
    "Test_ROC_AUC": "mean",
    "Test_PR_AUC": "mean",
    "Test_Balanced_Accuracy": "mean",
    "Test_F1": "mean",
    "TN": "sum",
    "FP": "sum",
    "FN": "sum",
    "TP": "sum"
})

results_df.to_csv(
    OUT_DIR / "three_fold_average_results.csv", index=False
)

print("\n========== THREE-FOLD AVERAGE RESULTS ==========")
print(results_df.to_string(index=False))

print("\nAll three folds completed successfully.")
print("Individual fold results:", (OUT_DIR / "three_fold_individual_results.csv").resolve())
print("Average results:", (OUT_DIR / "three_fold_average_results.csv").resolve())

# 13. SAVE ROC AND PRECISION-RECALL CURVES
for name, (labels, scores, preds, threshold) in predictions_by_model.items():
    safe = name.lower()
    fpr, tpr, roc_thresholds = roc_curve(labels, scores)
    precision, recall, _ = precision_recall_curve(labels, scores)
    auc, ap = roc_auc_score(labels, scores), average_precision_score(labels, scores)

    pd.DataFrame({"FPR": fpr, "TPR": tpr, "Threshold": roc_thresholds}).to_csv(
        OUT_DIR / f"roc_coordinates_{safe}.csv", index=False
    )
    pd.DataFrame({"Precision": precision, "Recall": recall}).to_csv(
        OUT_DIR / f"pr_coordinates_{safe}.csv", index=False
    )

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"HECKTOR ROC — {name}"); plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"roc_{safe}.png", dpi=300, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"AP={ap:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"HECKTOR Precision-Recall — {name}"); plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"pr_{safe}.png", dpi=300, bbox_inches="tight")
    plt.close()

# 14. SAVE SETTINGS AND COMPARISON RESULTS
settings = {
    "Dataset": "HECKTOR", "Model": MODEL_NAME, "Target": "HPV Status",
    "Features": ", ".join(FEATURES), "Number_of_Features": len(FEATURES),
    "Split": "3-fold stratified cross-validation", "Seed": SEED, "Epochs": EPOCHS,
    "Batch_Size": BATCH_SIZE, "Learning_Rate": LEARNING_RATE,
    "Weight_Decay": WEIGHT_DECAY, "Dropout": DROPOUT,
    "Max_Length": MAX_LENGTH, "Optimizer": "AdamW",
    "Class_Weighted_Loss": True, "GPT2_Backbone_Frozen": True,
    "Model_Type": "GPT-2 Classifier with LoRA",
    "LoRA_Rank": 6,
    "LoRA_Alpha": 12,
    "LoRA_Target_Modules": "c_attn",
    "LoRA_Dropout": 0.0,
    "Checkpoint_Selection": "Validation loss, ROC-AUC, PR-AUC, balanced accuracy",
    "Threshold_Selection": "Maximum validation balanced accuracy"
}
pd.DataFrame([settings]).to_csv(OUT_DIR / "settings.csv", index=False)

comparison_file = Path("model_roc_pr_auc_comparison.csv")
comparison = results_df[[
    "Dataset", "Model", "Checkpoint_Selected_By", "Best_Epoch",
    "Test_ROC_AUC", "Test_PR_AUC", "Test_Balanced_Accuracy", "Test_F1"
]]
if comparison_file.exists():
    old = pd.read_csv(comparison_file)
    combined = pd.concat([old, comparison], ignore_index=True, sort=False)
    combined = combined.drop_duplicates(
        subset=["Dataset", "Model", "Checkpoint_Selected_By"], keep="last"
    )
else:
    combined = comparison
combined.to_csv(comparison_file, index=False)

print("\nAll outputs saved to:", OUT_DIR.resolve())
print("Comparison CSV:", comparison_file.resolve())

PyTorch: 2.11.0+cu130
Transformers: 5.20.2
PEFT: 0.21.2
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Device: cuda


Downloading...
From: https://drive.google.com/uc?id=1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv
To: /content/HPV2025.xlsx
100%|██████████| 66.0k/66.0k [00:00<00:00, 184MB/s]


Dataset: (588, 9) | Invalid/missing target rows dropped: 138
HPV classes (0=negative, 1=positive):
 HPV_Label
0     58
1    530
Name: count, dtype: int64

Fold 1
Training samples: 392
Validation samples: 196
Training class counts: {0: 39, 1: 353}
Validation class counts: {0: 19, 1: 177}

Fold 2
Training samples: 392
Validation samples: 196
Training class counts: {0: 39, 1: 353}
Validation class counts: {0: 19, 1: 177}

Fold 3
Training samples: 392
Validation samples: 196
Training class counts: {0: 38, 1: 354}
Validation class counts: {0: 20, 1: 176}

3-fold cross-validation splits created successfully.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2760: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 221,184 || all params: 124,660,992 || trainable%: 0.1774
Class weights [negative, positive]: [5.025641  0.5552408]
Total trainable parameters (LoRA + classifier): 321410
Fold 2 | Epoch 50/500 | Train Loss=0.5920 | Val Loss=0.5713 | ROC-AUC=0.7572 | PR-AUC=0.9691 | BA=0.7281 | Threshold=0.6491
Fold 2 | Epoch 100/500 | Train Loss=0.4585 | Val Loss=1.1889 | ROC-AUC=0.6742 | PR-AUC=0.9491 | BA=0.6913 | Threshold=0.6997
Fold 2 | Epoch 150/500 | Train Loss=0.3619 | Val Loss=1.8342 | ROC-AUC=0.6406 | PR-AUC=0.9457 | BA=0.6536 | Threshold=0.9998
Fold 2 | Epoch 200/500 | Train Loss=0.3014 | Val Loss=1.8569 | ROC-AUC=0.6408 | PR-AUC=0.9440 | BA=0.6650 | Threshold=0.9797
Fold 2 | Epoch 250/500 | Train Loss=0.2580 | Val Loss=2.5479 | ROC-AUC=0.6326 | PR-AUC=0.9390 | BA=0.6735 | Threshold=0.9885
Fold 2 | Epoch 300/500 | Train Loss=0.2522 | Val Loss=3.1949 | ROC-AUC=0.5944 | PR-AUC=0.9341 | BA=0.6246 | Threshold=0.9964
Fold 2 | Epoch 350/500 | Train Loss=0.7163 | Val Loss=0.6864 | 

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2760: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 221,184 || all params: 124,660,992 || trainable%: 0.1774
Class weights [negative, positive]: [5.1578946 0.5536723]
Total trainable parameters (LoRA + classifier): 321410
Fold 3 | Epoch 50/500 | Train Loss=0.6969 | Val Loss=0.7090 | ROC-AUC=0.6489 | PR-AUC=0.9470 | BA=0.6585 | Threshold=0.4886
Fold 3 | Epoch 100/500 | Train Loss=0.6949 | Val Loss=0.7092 | ROC-AUC=0.6190 | PR-AUC=0.9355 | BA=0.6125 | Threshold=0.4960
Fold 3 | Epoch 150/500 | Train Loss=0.6936 | Val Loss=0.7113 | ROC-AUC=0.2884 | PR-AUC=0.8512 | BA=0.5091 | Threshold=0.5188
Fold 3 | Epoch 200/500 | Train Loss=0.6939 | Val Loss=0.7097 | ROC-AUC=0.3521 | PR-AUC=0.8678 | BA=0.5170 | Threshold=0.5035
Fold 3 | Epoch 250/500 | Train Loss=0.6939 | Val Loss=0.7094 | ROC-AUC=0.2226 | PR-AUC=0.8413 | BA=0.5119 | Threshold=0.4724
Fold 3 | Epoch 300/500 | Train Loss=0.6930 | Val Loss=0.7096 | ROC-AUC=0.2835 | PR-AUC=0.8543 | BA=0.5176 | Threshold=0.5027
Fold 3 | Epoch 350/500 | Train Loss=0.6937 | Val Loss=0.7098 | 